In [30]:
import pandas as pd
import numpy as np
from sklearn.metrics.pairwise import cosine_similarity

df = pd.read_csv('../data/cleaned_dataset.csv')
print(df.shape)
df.head()

(114000, 12)


,track_name,artists,track_genre,popularity,energy,valence,danceability,tempo,acousticness,instrumentalness,liveness,speechiness
0,Comedy,Gen Hoshino,acoustic,73,0.4610,0.718593,0.686294,0.361245,0.032329,0.000001,0.3580,0.148187
1,Ghost - Acoustic,Ben Woodward,acoustic,55,0.1660,0.268342,0.426396,0.318397,0.927711,0.000006,0.1010,0.079067
2,To Begin Again,Ingrid Michaelson;ZAYN,acoustic,57,0.3590,0.120603,0.444670,0.313643,0.210843,0.000000,0.1170,0.057720
3,Can't Help Falling In Love,Kina Grannis,acoustic,71,0.0596,0.143719,0.270051,0.746758,0.908635,0.000071,0.1320,0.037617
4,Hold On,Chord Overstreet,acoustic,82,0.4430,0.167839,0.627411,0.492863,0.470884,0.000000,0.0829,0.054508


In [31]:
feature_cols = ['energy', 'valence', 'danceability', 'tempo',
                'acousticness', 'instrumentalness', 'liveness', 'speechiness']

# Extract just the numbers into a matrix
song_matrix = df[feature_cols].values

print(f"Song matrix shape: {song_matrix.shape}")

Song matrix shape: (114000, 8)


In [32]:
def recommend_songs(song_name, df, song_matrix, n=10):
    matches = df[df['track_name'].str.lower() == song_name.lower()]
    
    if matches.empty:
        print(f"Song '{song_name}' not found in dataset.")
        return None
    
    song_idx = matches.loc[matches['popularity'].idxmax()].name
    print(f"Found: {df.loc[song_idx, 'track_name']} by {df.loc[song_idx, 'artists']}")
    
    song_vector = song_matrix[song_idx].reshape(1, -1)
    similarities = cosine_similarity(song_vector, song_matrix)[0]
    similar_indices = np.argsort(similarities)[::-1][1:n*3]  # grab extra to account for drops
    
    results = df.iloc[similar_indices][['track_name', 'artists', 'track_genre', 'popularity']].copy()
    results['similarity_score'] = similarities[similar_indices].round(4)

    # Drop duplicate songs, keeping the most popular version
    results = results.sort_values('popularity', ascending=False)
    results = results.drop_duplicates(subset=['track_name', 'artists'])

    # Now sort by similarity and trim to n
    results = results.sort_values('similarity_score', ascending=False).head(n)
    results = results.reset_index(drop=True)
    
    return results

In [33]:
recommendations = recommend_songs("What Makes You Beautiful", df, song_matrix, n=10)
print(recommendations)

Found: What Makes You Beautiful by One Direction
                               track_name             artists track_genre  \
0                           Luften är fri      2 Blyga Läppar     swedish   
1                      Give It To Me Baby          Rick James        soul   
2         You're Only Human (Second Wind)          Billy Joel       piano   
3                                  Melanż             MiłyPan       disco   
4                           Fast Steppin'      Michelle Ayers      garage   
5                  Lady - Hear Me Tonight               Modjo       disco   
6  Pandangan Pertama (feat. Nirina Zubir)  Slank;Nirina Zubir       blues   
7          All 4 Nothing (I'm So In Love)                Lauv     electro   
8                              Angel Eyes                Lime       disco   
9                               Luv 4 Luv             Robin S       disco   

   popularity  similarity_score  
0          45            1.0000  
1           0            0.9998  
2

In [34]:
print(df.columns.tolist())

['track_name', 'artists', 'track_genre', 'popularity', 'energy', 'valence', 'danceability', 'tempo', 'acousticness', 'instrumentalness', 'liveness', 'speechiness']


In [96]:
MOOD_PROFILES = {
    "happy":     {"valence": 0.8, "energy": 0.7, "danceability": 0.7},
    "sad":       {"valence": 0.2, "energy": 0.3, "danceability": 0.3},
    "angry":     {"valence": 0.7, "energy": 0.9, "danceability": 0.5},
    "chill":     {"valence": 0.5, "energy": 0.2, "danceability": 0.4},
    "energetic": {"valence": 0.6, "energy": 0.9, "danceability": 0.8},
    "romantic":  {"valence": 0.6, "energy": 0.3, "danceability": 0.4},
    "dance":     {"valence": 0.7, "energy": 0.8, "danceability": 0.95},
    "focus":     {"valence": 0.4, "energy": 0.4, "instrumentalness": 0.8},
}

# Genre → filter (these are actual values in your track_genre column)
GENRE_LIST = df['track_genre'].unique().tolist()
print(GENRE_LIST)  

['acoustic', 'afrobeat', 'alt-rock', 'alternative', 'ambient', 'anime', 'black-metal', 'bluegrass', 'blues', 'brazil', 'breakbeat', 'british', 'cantopop', 'chicago-house', 'children', 'chill', 'classical', 'club', 'comedy', 'country', 'dance', 'dancehall', 'death-metal', 'deep-house', 'detroit-techno', 'disco', 'disney', 'drum-and-bass', 'dub', 'dubstep', 'edm', 'electro', 'electronic', 'emo', 'folk', 'forro', 'french', 'funk', 'garage', 'german', 'gospel', 'goth', 'grindcore', 'groove', 'grunge', 'guitar', 'happy', 'hard-rock', 'hardcore', 'hardstyle', 'heavy-metal', 'hip-hop', 'honky-tonk', 'house', 'idm', 'indian', 'indie-pop', 'indie', 'industrial', 'iranian', 'j-dance', 'j-idol', 'j-pop', 'j-rock', 'jazz', 'k-pop', 'kids', 'latin', 'latino', 'malay', 'mandopop', 'metal', 'metalcore', 'minimal-techno', 'mpb', 'new-age', 'opera', 'pagode', 'party', 'piano', 'pop-film', 'pop', 'power-pop', 'progressive-house', 'psych-rock', 'punk-rock', 'punk', 'r-n-b', 'reggae', 'reggaeton', 'rock-n

In [97]:
NON_ENGLISH_GENRES = [
    'brazil', 'cantopop', 'forro', 'french', 'german', 'iranian',
    'j-dance', 'j-idol', 'j-pop', 'j-rock', 'k-pop', 'latin',
    'latino', 'malay', 'mandopop', 'mpb', 'pagode', 'reggaeton',
    'romance', 'salsa', 'samba', 'sertanejo', 'spanish', 'swedish',
    'tango', 'turkish'
]

In [98]:
def recommend_by_mood_and_genre(mood, genre=None, english_only=False, df=df, song_matrix=song_matrix, n=10):

    if mood.lower() not in MOOD_PROFILES:
        print(f"Unknown mood '{mood}'. Available: {list(MOOD_PROFILES.keys())}")
        return None

    pref = {
        "energy": 0.5, "valence": 0.5, "danceability": 0.5,
        "tempo": 0.5, "acousticness": 0.5, "instrumentalness": 0.5,
        "liveness": 0.5, "speechiness": 0.5,
    }
    pref.update(MOOD_PROFILES[mood.lower()])

    feature_cols = ['energy', 'valence', 'danceability', 'tempo',
                    'acousticness', 'instrumentalness', 'liveness', 'speechiness']

    user_vector = np.array([[pref[f] for f in feature_cols]])

    pool = df.copy()
    pool_matrix = song_matrix

    if genre:
        pool = pool[pool['track_genre'].str.lower() == genre.lower()]
        pool_matrix = song_matrix[pool.index]

    if english_only:
        pool = pool[~pool['track_genre'].isin(NON_ENGLISH_GENRES)]
        pool_matrix = song_matrix[pool.index]

    if pool.empty:
        print("No songs found matching those filters.")
        return None

    similarities = cosine_similarity(user_vector, pool_matrix)[0]

    # Grab extra candidates to account for deduplication
    top_indices = np.argsort(similarities)[::-1][:n*3]

    results = pool.iloc[top_indices][['track_name', 'artists', 'track_genre', 'popularity']].copy()
    results['similarity_score'] = similarities[top_indices].round(4)

    # Deduplicate keeping most popular version
    results = results.sort_values('popularity', ascending=False)
    results = results.drop_duplicates(subset=['track_name', 'artists'])

    # Sort by similarity and trim to n
    results = results.sort_values('similarity_score', ascending=False).head(n)
    results = results.reset_index(drop=True)

    return results

In [99]:
recommend_by_mood_and_genre(mood="angry", genre="pop")

,track_name,artists,track_genre,popularity,similarity_score
0,Que Raro,Feid;J Balvin,pop,2,0.9250
1,Qismat,Ammy Virk,pop,62,0.9241
2,Cradles,Sub Urban,pop,76,0.9190
3,Ganjai Poovu,Yuvan Shankar Raja;Sarath Santosh,pop,47,0.9180
4,Temporary pyar,Kaka,pop,65,0.9178
5,Baarish Ki Jaaye,B Praak;Nawazuddin Siddiqui;Sunanda Sharma,pop,63,0.9151
6,"Hey Mama (feat. Nicki Minaj, Bebe Rexha & Afro...",David Guetta;Afrojack;Bebe Rexha;Nicki Minaj,pop,75,0.9138
7,Otha Thamarai - Original Soundtrack,Bala;Nixen;Sandy Sandellow,pop,68,0.9099
8,Nuestra Canción,Feid,pop,0,0.9092
9,Priceless,Bhalwaan;Signature By SB,pop,65,0.9076


In [100]:
recommendations = recommend_by_mood_and_genre(mood="happy", genre="pop")
print(recommendations['track_genre'].unique())

['pop']
